# Wind Statistics

### Introduction:

The data have been modified to contain some missing values, identified by NaN.  
Using pandas should make this exercise
easier, in particular for the bonus question.

You should be able to perform all of these operations without using
a for loop or other looping construct.


1. The data in 'wind.data' has the following format:

In [9]:
"""
Yr Mo Dy   RPT   VAL   ROS   KIL   SHA   BIR   DUB   CLA   MUL   CLO   BEL   MAL
61  1  1 15.04 14.96 13.17  9.29   NaN  9.87 13.67 10.25 10.83 12.58 18.50 15.04
61  1  2 14.71   NaN 10.83  6.50 12.62  7.67 11.50 10.04  9.79  9.67 17.54 13.83
61  1  3 18.50 16.88 12.33 10.13 11.17  6.17 11.25   NaN  8.50  7.67 12.75 12.71
"""

'\nYr Mo Dy   RPT   VAL   ROS   KIL   SHA   BIR   DUB   CLA   MUL   CLO   BEL   MAL\n61  1  1 15.04 14.96 13.17  9.29   NaN  9.87 13.67 10.25 10.83 12.58 18.50 15.04\n61  1  2 14.71   NaN 10.83  6.50 12.62  7.67 11.50 10.04  9.79  9.67 17.54 13.83\n61  1  3 18.50 16.88 12.33 10.13 11.17  6.17 11.25   NaN  8.50  7.67 12.75 12.71\n'

   The first three columns are year, month and day.  The
   remaining 12 columns are average windspeeds in knots at 12
   locations in Ireland on that day.   

   More information about the dataset go [here](wind.desc).

### Step 1. Import the necessary libraries

In [10]:
import pandas as pd
import datetime


### Step 2. Import the dataset from this [address](https://raw.githubusercontent.com/thieu1995/csv-files/main/data/pandas/wind.data)

### Step 3. Assign it to a variable called data and replace the first 3 columns by a proper datetime index.

In [11]:
data = pd.read_csv('wind_stats.tsv', sep='\s+')
date_strings = data['Yr'].astype(str) + '-' + data['Mo'].astype(str) + '-' + data['Dy'].astype(str)
data['Yr_Mo_Dy'] = pd.to_datetime(date_strings, format='%y-%m-%d')
data = data.drop(['Yr', 'Mo', 'Dy'], axis=1)
cols = ['Yr_Mo_Dy'] + [c for c in data.columns if c != 'Yr_Mo_Dy']
data = data[cols]
print(data.head())

    Yr_Mo_Dy    RPT    VAL    ROS    KIL    SHA   BIR    DUB    CLA    MUL  \
0 2061-01-01  15.04  14.96  13.17   9.29    NaN  9.87  13.67  10.25  10.83   
1 2061-01-02  14.71    NaN  10.83   6.50  12.62  7.67  11.50  10.04   9.79   
2 2061-01-03  18.50  16.88  12.33  10.13  11.17  6.17  11.25    NaN   8.50   
3 2061-01-04  10.58   6.63  11.75   4.58   4.54  2.88   8.63   1.79   5.83   
4 2061-01-05  13.33  13.25  11.42   6.17  10.71  8.21  11.92   6.54  10.92   

     CLO    BEL    MAL  
0  12.58  18.50  15.04  
1   9.67  17.54  13.83  
2   7.67  12.75  12.71  
3   5.88   5.46  10.88  
4  10.34  12.92  11.83  


### Step 4. Year 2061? Do we really have data from this year? Create a function to fix it and apply it.

In [12]:
import datetime
def fix_century(x):
    year = x.year - 100 if x.year > 1989 else x.year
    return datetime.date(year, x.month, x.day)
data['Yr_Mo_Dy'] = data['Yr_Mo_Dy'].apply(fix_century)
data['Yr_Mo_Dy'] = pd.to_datetime(data['Yr_Mo_Dy'])
print(data.head())

    Yr_Mo_Dy    RPT    VAL    ROS    KIL    SHA   BIR    DUB    CLA    MUL  \
0 1961-01-01  15.04  14.96  13.17   9.29    NaN  9.87  13.67  10.25  10.83   
1 1961-01-02  14.71    NaN  10.83   6.50  12.62  7.67  11.50  10.04   9.79   
2 1961-01-03  18.50  16.88  12.33  10.13  11.17  6.17  11.25    NaN   8.50   
3 1961-01-04  10.58   6.63  11.75   4.58   4.54  2.88   8.63   1.79   5.83   
4 1961-01-05  13.33  13.25  11.42   6.17  10.71  8.21  11.92   6.54  10.92   

     CLO    BEL    MAL  
0  12.58  18.50  15.04  
1   9.67  17.54  13.83  
2   7.67  12.75  12.71  
3   5.88   5.46  10.88  
4  10.34  12.92  11.83  


### Step 5. Set the right dates as the index. Pay attention at the data type, it should be datetime64[ns].

In [13]:
data = data.set_index('Yr_Mo_Dy')
print(data.index)

DatetimeIndex(['1961-01-01', '1961-01-02', '1961-01-03', '1961-01-04',
               '1961-01-05', '1961-01-06', '1961-01-07', '1961-01-08',
               '1961-01-09', '1961-01-10',
               ...
               '1978-12-22', '1978-12-23', '1978-12-24', '1978-12-25',
               '1978-12-26', '1978-12-27', '1978-12-28', '1978-12-29',
               '1978-12-30', '1978-12-31'],
              dtype='datetime64[s]', name='Yr_Mo_Dy', length=6574, freq=None)


### Step 6. Compute how many values are missing for each location over the entire record.  
#### They should be ignored in all calculations below. 

In [14]:
# Hàm isnull() tạo ra bảng True/False cho các giá trị thiếu, sum() sẽ cộng tổng theo từng cột
missing_per_loc = data.isnull().sum()
print(missing_per_loc)

RPT    6
VAL    3
ROS    2
KIL    5
SHA    2
BIR    0
DUB    3
CLA    2
MUL    3
CLO    1
BEL    0
MAL    4
dtype: int64


### Step 7. Compute how many non-missing values there are in total.

In [15]:
# notnull() đếm giá trị hợp lệ, sum() lần 1 tính theo cột, sum() lần 2 tính tổng của tất cả các cột
total_valid_values = data.notnull().sum().sum()
print("Tổng số giá trị hợp lệ:", total_valid_values)

Tổng số giá trị hợp lệ: 78857


### Step 8. Calculate the mean windspeeds of the windspeeds over all the locations and all the times.
#### A single number for the entire dataset.

In [16]:
overall_mean = data.stack().mean()
print("Tốc độ gió trung bình toàn bộ:", overall_mean)

Tốc độ gió trung bình toàn bộ: 10.227883764282181


### Step 9. Create a DataFrame called loc_stats and calculate the min, max and mean windspeeds and standard deviations of the windspeeds at each location over all the days 

#### A different set of numbers for each location.

In [17]:
loc_stats = data.agg(['min', 'max', 'mean', 'std']).T
print(loc_stats)

      min    max       mean       std
RPT  0.67  35.80  12.362987  5.618413
VAL  0.21  33.37  10.644314  5.267356
ROS  1.50  33.84  11.660526  5.008450
KIL  0.00  28.46   6.306468  3.605811
SHA  0.13  37.54  10.455834  4.936125
BIR  0.00  26.16   7.092254  3.968683
DUB  0.00  30.37   9.797343  4.977555
CLA  0.00  31.08   8.495053  4.499449
MUL  0.00  25.88   8.493590  4.166872
CLO  0.04  28.21   8.707332  4.503954
BEL  0.13  42.38  13.121007  5.835037
MAL  0.67  42.54  15.599079  6.699794


### Step 10. Create a DataFrame called day_stats and calculate the min, max and mean windspeed and standard deviations of the windspeeds across all the locations at each day.

#### A different set of numbers for each day.

In [18]:
day_stats = data.agg(['min', 'max', 'mean', 'std'], axis=1)
print(day_stats.head())

             min    max       mean       std
Yr_Mo_Dy                                    
1961-01-01  9.29  18.50  13.018182  2.808875
1961-01-02  6.50  17.54  11.336364  3.188994
1961-01-03  6.17  18.50  11.641818  3.681912
1961-01-04  1.79  11.75   6.619167  3.198126
1961-01-05  6.17  13.33  10.630000  2.445356


### Step 11. Find the average windspeed in January for each location.  
#### Treat January 1961 and January 1962 both as January.

In [19]:
jan_avg = data[data.index.month == 1].mean()
print(jan_avg)

RPT    14.847325
VAL    12.914560
ROS    13.299624
KIL     7.199498
SHA    11.667734
BIR     8.054839
DUB    11.819355
CLA     9.512047
MUL     9.543208
CLO    10.053566
BEL    14.550520
MAL    18.028763
dtype: float64


### Step 12. Downsample the record to a yearly frequency for each location.

In [20]:
yearly_data = data.resample('YE').mean()
print(yearly_data.head())

                  RPT        VAL        ROS       KIL        SHA       BIR  \
Yr_Mo_Dy                                                                     
1961-12-31  12.299583  10.351796  11.362369  6.958227  10.881763  7.729726   
1962-12-31  12.246923  10.110438  11.732712  6.960440  10.657918  7.393068   
1963-12-31  12.813452  10.836986  12.541151  7.330055  11.724110  8.434712   
1964-12-31  12.363661  10.920164  12.104372  6.787787  11.454481  7.570874   
1965-12-31  12.451370  11.075534  11.848767  6.858466  11.024795  7.478110   

                  DUB        CLA       MUL        CLO        BEL        MAL  
Yr_Mo_Dy                                                                     
1961-12-31   9.733923   8.858788  8.647652   9.835577  13.502795  13.680773  
1962-12-31  11.020712   8.793753  8.316822   9.676247  12.930685  14.323956  
1963-12-31  11.075699  10.336548  8.903589  10.224438  13.638877  14.999014  
1964-12-31  10.259153   9.467350  7.789016  10.207951  13.74054

### Step 13. Downsample the record to a monthly frequency for each location.

In [21]:
monthly_data = data.resample('ME').mean()
print(monthly_data.head())

                  RPT        VAL        ROS       KIL        SHA        BIR  \
Yr_Mo_Dy                                                                      
1961-01-31  14.841333  11.988333  13.431613  7.736774  11.072759   8.588065   
1961-02-28  16.269286  14.975357  14.441481  9.230741  13.852143  10.937500   
1961-03-31  10.890000  11.296452  10.752903  7.284000  10.509355   8.866774   
1961-04-30  10.722667   9.427667   9.998000  5.830667   8.435000   6.495000   
1961-05-31   9.860968   8.850000  10.818065  5.905333   9.490323   6.574839   

                  DUB        CLA        MUL        CLO        BEL        MAL  
Yr_Mo_Dy                                                                      
1961-01-31  11.184839   9.245333   9.085806  10.107419  13.880968  14.703226  
1961-02-28  11.890714  11.846071  11.821429  12.714286  18.583214  15.411786  
1961-03-31   9.644194   9.829677  10.294138  11.251935  16.410968  15.720000  
1961-04-30   6.925333   7.094667   7.342333   7.237

### Step 14. Downsample the record to a weekly frequency for each location.

In [22]:
weekly_data = data.resample('W').mean()
print(weekly_data.head())

                  RPT        VAL        ROS        KIL        SHA        BIR  \
Yr_Mo_Dy                                                                       
1961-01-01  15.040000  14.960000  13.170000   9.290000        NaN   9.870000   
1961-01-08  13.541429  11.486667  10.487143   6.417143   9.474286   6.435714   
1961-01-15  12.468571   8.967143  11.958571   4.630000   7.351429   5.072857   
1961-01-22  13.204286   9.862857  12.982857   6.328571   8.966667   7.417143   
1961-01-29  19.880000  16.141429  18.225714  12.720000  17.432857  14.828571   

                  DUB        CLA        MUL        CLO        BEL        MAL  
Yr_Mo_Dy                                                                      
1961-01-01  13.670000  10.250000  10.830000  12.580000  18.500000  15.040000  
1961-01-08  11.061429   6.616667   8.434286   8.497143  12.481429  13.238571  
1961-01-15   7.535714   6.820000   5.712857   7.571429  11.125714  11.024286  
1961-01-22   9.257143   7.875714   7.145714 

### Step 15. Calculate the min, max and mean windspeeds and standard deviations of the windspeeds across all locations for each week (assume that the first week starts on January 2 1961) for the first 52 weeks.

In [23]:
weekly_resampled = data.resample('W-MON').mean()
weekly_52 = weekly_resampled.loc['1961-01-02':].head(52)
weekly_stats = weekly_52.agg(['min', 'max', 'mean', 'std'], axis=1)
print(weekly_stats)

                  min        max       mean       std
Yr_Mo_Dy                                             
1961-01-02   7.895000  18.020000  12.311667  2.909304
1961-01-09   6.167143  13.458571   9.640496  2.575846
1961-01-16   4.624286  13.017143   8.391310  2.842770
1961-01-23   7.150000  13.392857   9.925556  2.183014
1961-01-30  12.357143  22.340000  16.729028  2.988786
1961-02-06   9.311429  18.582857  12.918214  3.030879
1961-02-13  10.770000  22.152857  15.569405  3.305013
1961-02-20   8.281429  19.234286  12.590655  2.902097
1961-02-27   8.315714  17.005714  12.920714  2.492463
1961-03-06   7.300000  16.444286  10.751290  2.313917
1961-03-13   7.494286  18.057143  11.617877  2.945308
1961-03-20   7.815714  18.577143  11.831429  3.109227
1961-03-27   5.505714  17.312857   9.403571  3.148063
1961-04-03   7.528333  14.024286  10.773651  1.728760
1961-04-10   5.798571  12.547143   8.938095  2.207989
1961-04-17   4.458571   9.828571   6.573690  1.837791
1961-04-24   8.227143  13.59